# J03 — Courbe de rétention, hystérésis et fonctions de pédotransfert

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 3 : ajustement VG/BC, points remarquables et réserve utile, pédotransfert, hystérésis*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Les données sont dans `data/`. Unités du cours : $h$ en cm (négatif en non saturé), $\theta$ en m³/m³, $K_s$ en cm/j.
Les sols de référence sont ceux de Carsel & Parrish (1988).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, special

plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.3})

# Carsel & Parrish (1988) : θr, θs, α (1/cm), n, Ks (cm/j)
CP = {"sable": (0.045, 0.43, 0.145, 2.68, 712.8), "loam sableux": (0.065, 0.41, 0.075, 1.89, 106.1),
      "loam": (0.078, 0.43, 0.036, 1.56, 24.96), "loam limoneux": (0.067, 0.45, 0.020, 1.41, 10.8),
      "loam argileux": (0.095, 0.41, 0.019, 1.31, 6.24), "argile": (0.068, 0.38, 0.008, 1.09, 4.8)}

def vg(h, tr, ts, a, n):
    """van Genuchten (1980) : θ(h), m = 1 - 1/n."""
    m = 1 - 1 / n
    return tr + (ts - tr) * (1 + (a * np.abs(h)) ** n) ** (-m)

def vg_C(h, tr, ts, a, n):
    """Capacité capillaire C(h) = dθ/dh (1/cm) du modèle de van Genuchten."""
    m = 1 - 1 / n
    ah = a * np.abs(h)
    return (ts - tr) * a * n * m * ah ** (n - 1) * (1 + ah ** n) ** (-m - 1)

def bc(h, tr, ts, hb, lam):
    """Brooks & Corey (1964) : θ(h) avec pression d'entrée d'air hb (> 0, cm) et indice λ."""
    h = np.abs(h)
    return np.where(h <= hb, ts, tr + (ts - tr) * (hb / h) ** lam)

## Exercice 1 — Ajustement des modèles de van Genuchten et de Brooks–Corey  *(≈ 20 min)*

`data/J03_retention_mesures.csv` : couples $(h, \theta)$ mesurés en drainage (9 points, bruit expérimental) sur un sable, un loam
et une argile.

1. Pour chaque sol, ajuster le modèle de van Genuchten ($\theta_r$, $\theta_s$, $\alpha$, $n$) et celui de Brooks–Corey
   ($\theta_r$, $\theta_s$, $h_b$, $\lambda$) avec `scipy.optimize.curve_fit`, en imposant des bornes physiques
   ($0 \le \theta_r \le 0{,}25$, $0{,}3 \le \theta_s \le 0{,}6$, $10^{-4} \le \alpha \le 1$, $1{,}05 \le n \le 8$ ;
   $1 \le h_b \le 500$, $0{,}05 \le \lambda \le 3$) et une initialisation raisonnable.
2. Calculer le RMSE et le $R^2$ de chaque ajustement ; présenter un tableau (paramètres, RMSE, $R^2$) et le comparer aux
   paramètres de Carsel & Parrish (dictionnaire `CP`).
3. Tracer, pour les trois sols, les points mesurés et les deux modèles ($|h|$ en échelle log), puis la capacité capillaire $C(h)$
   du modèle VG (échelle log–log).
4. Extraire de la matrice de covariance les écarts-types des paramètres VG et la corrélation entre $\alpha$ et $n$.

In [ ]:
mes = pd.read_csv("data/J03_retention_mesures.csv")
hh = -np.logspace(-0.5, 4.5, 300)

def rmse_r2(y, yhat):
    # À COMPLÉTER
    pass

res = {}
for sol, g in mes.groupby("sol", sort=False):
    h, th = g["h_cm"].to_numpy(), g["theta"].to_numpy()
    p_vg, cov_vg = optimize.curve_fit(vg, h, th, p0=[0.05, 0.45, 0.02, 1.5],
                                      bounds=([0, 0.3, 1e-4, 1.05], [0.25, 0.6, 1, 8]))
    p_bc, cov_bc = # À COMPLÉTER
    # RMSE, R², tracés, covariance

## Exercice 2 — Points remarquables, réserve utile et réserve facilement utilisable  *(≈ 10 min)*

Avec les paramètres VG ajustés à l'exercice 1 :

1. Calculer $\theta$ à $-100$, $-330$ et $-15\,000$ cm pour chaque sol ; en déduire la réserve utile par mètre de sol,
   $\mathrm{RU} = (\theta_{cc} - \theta_{pf}) \times 1000$ mm, selon les deux conventions de capacité au champ ($-100$ et $-330$ cm).
2. Pour un enracinement $Z = 60$ cm et $p = 0{,}5$, calculer la RFU (mm) et le nombre de jours d'autonomie sans pluie pour
   $\mathrm{ET}_c = 5$ mm/j (convention $-330$ cm pour le loam et l'argile, $-100$ cm pour le sable).
3. Représenter en barres $\theta_{-100}$, $\theta_{-330}$, $\theta_{-15000}$ et $\theta_s$ pour les trois sols.

In [ ]:
rows = []
for sol, f in fits.items():
    p = f["vg"]
    t100, t330, tpf = # À COMPLÉTER
    # RU100, RU330 (mm/m), RFU (Z = 60 cm, p = 0,5), jours d'autonomie
pts = pd.DataFrame(rows).set_index("sol")
display(pts)

## Exercice 3 — Fonctions de pédotransfert : Saxton & Rawls vs Carsel & Parrish vs Rosetta  *(≈ 20 min)*

1. Implémenter `saxton_rawls(S, C, OM)` (S = sable et C = argile en fractions 0–1, OM en % masse) selon Saxton & Rawls (2006) :

   $\theta_{1500}^t = -0{,}024S + 0{,}487C + 0{,}006\,OM + 0{,}005\,S\,OM - 0{,}013\,C\,OM + 0{,}068\,SC + 0{,}031$ ;
   $\theta_{1500} = \theta_{1500}^t + (0{,}14\,\theta_{1500}^t - 0{,}02)$

   $\theta_{33}^t = -0{,}251S + 0{,}195C + 0{,}011\,OM + 0{,}006\,S\,OM - 0{,}027\,C\,OM + 0{,}452\,SC + 0{,}299$ ;
   $\theta_{33} = \theta_{33}^t + (1{,}283\,(\theta_{33}^t)^2 - 0{,}374\,\theta_{33}^t - 0{,}015)$

   $\theta_{S-33}^t = 0{,}278S + 0{,}034C + 0{,}022\,OM - 0{,}018\,S\,OM - 0{,}027\,C\,OM - 0{,}584\,SC + 0{,}078$ ;
   $\theta_{S-33} = \theta_{S-33}^t + (0{,}636\,\theta_{S-33}^t - 0{,}107)$

   $\theta_s = \theta_{33} + \theta_{S-33} - 0{,}097S + 0{,}043$ ; $\lambda = (\ln\theta_{33} - \ln\theta_{1500})/(\ln 1500 - \ln 33)$ ;
   $K_s\,[\mathrm{mm/h}] = 1930\,(\theta_s - \theta_{33})^{3-\lambda}$ ;

   pression d'entrée d'air (kPa) : $\psi_e^t = -21{,}67S - 27{,}93C - 81{,}97\,\theta_{S-33} + 71{,}12\,S\,\theta_{S-33} + 8{,}29\,C\,\theta_{S-33} + 14{,}05\,SC + 27{,}16$,
   $\psi_e = \psi_e^t + (0{,}02\,(\psi_e^t)^2 - 0{,}113\,\psi_e^t - 0{,}70)$.

   La courbe complète est : $\theta = \theta_s$ pour $\psi < \psi_e$ ; linéaire entre $(\psi_e, \theta_s)$ et $(33, \theta_{33})$ ;
   $\psi = A\,\theta^{-B}$ au-delà de 33 kPa, avec $B = 1/\lambda$ et $A = \exp(\ln 33 + B\ln\theta_{33})$.
2. `data/J03_textures.csv` (six sols A–F, texture, MO, $\rho_b$, classe USDA) : calculer les prédictions de Saxton–Rawls
   et les comparer, pour chaque sol, à $\theta(-330)$, $\theta(-15000)$ et $K_s$ (i) des paramètres de Carsel & Parrish de la classe
   et (ii) des paramètres Rosetta fournis dans `data/J03_rosetta_H3.csv`. Tableau récapitulatif.
3. Tracer $\theta(h)$ des trois sources pour les sols C (loam) et F (argile). Calculer, pour chaque sol, l'écart type entre les trois
   sources de la RU (mm/m) et de $\log_{10} K_s$ : quelle grandeur est la plus incertaine ?

In [ ]:
def saxton_rawls(S, C, OM):
    """Saxton & Rawls (2006). S, C : fractions (0-1) ; OM : % masse. Retourne un dict."""
    t1500t = -0.024*S + 0.487*C + 0.006*OM + 0.005*S*OM - 0.013*C*OM + 0.068*S*C + 0.031
    t1500 = t1500t + (0.14*t1500t - 0.02)
    # À COMPLÉTER : t33, tS33, tS, lam, Ks (cm/j), psi_e
    return dict(theta_1500=t1500)

def theta_sr(psi_kPa, p):
    """Courbe θ(ψ) de Saxton-Rawls (ψ > 0 en kPa) : plateau, segment linéaire, loi de puissance."""
    # À COMPLÉTER
    pass

tex = pd.read_csv("data/J03_textures.csv")
ros = pd.read_csv("data/J03_rosetta_H3.csv")

## Exercice 4 — Hystérésis : courbes principales et courbes de balayage (Kool & Parker, 1987)  *(≈ 10 min)*

Loam sableux : $\theta_r = 0{,}065$, $\theta_s^d = 0{,}41$, $\alpha_d = 0{,}075$ cm$^{-1}$, $n = 1{,}89$ ; humectation : $\alpha_w = 2\alpha_d$,
$\theta_s^w = 0{,}37$ (air piégé). Avec $S_e^d(h) = [1+(\alpha_d|h|)^n]^{-m}$ et $S_e^w(h) = [1+(\alpha_w|h|)^n]^{-m}$ :

* courbe de balayage de **drainage** depuis le point de renversement $(h_\Delta, \theta_\Delta)$ :
  $\theta(h) = \theta_r + (\theta_s' - \theta_r)\,S_e^d(h)$ avec $\theta_s' = \theta_r + (\theta_\Delta - \theta_r)/S_e^d(h_\Delta)$ ;
* courbe de balayage d'**humectation** : $\theta(h) = \theta_r' + (\theta_s^w - \theta_r')\,S_e^w(h)$ avec
  $\theta_r' = [\theta_\Delta - \theta_s^w S_e^w(h_\Delta)]/[1 - S_e^w(h_\Delta)]$.

1. Programmer `theta_d(h)`, `theta_w(h)`, `balayage_drainage(h, h_delta, theta_delta)` et `balayage_humectation(h, h_delta, theta_delta)`.
2. Tracer les deux courbes principales, puis le cycle : drainage principal de 0 à $-200$ cm, humectation (balayage) de $-200$ à $-20$ cm,
   drainage (balayage) de $-20$ à $-500$ cm.
3. Donner $\theta$ à $h = -50$ cm sur les deux courbes principales et sur les deux branches de balayage ; quelle erreur commet-on
   en utilisant la seule courbe de drainage ?

In [ ]:
tr, ts_d, ad, n = 0.065, 0.41, 0.075, 1.89
aw, ts_w = 2 * ad, 0.37
m = 1 - 1 / n
Se_d = lambda h: (1 + (ad * np.abs(h)) ** n) ** (-m)
Se_w = lambda h: (1 + (aw * np.abs(h)) ** n) ** (-m)
theta_d = lambda h: tr + (ts_d - tr) * Se_d(h)
theta_w = lambda h: tr + (ts_w - tr) * Se_w(h)

def balayage_drainage(h, h_delta, theta_delta):
    # À COMPLÉTER
    pass

def balayage_humectation(h, h_delta, theta_delta):
    # À COMPLÉTER
    pass

## Exercice 5 — Bonus — modèles de Kosugi et de Durner, critère AIC  *(≈ facultatif)*

Sur l'argile de l'exercice 1, ajuster le modèle de Kosugi (1996), $S_e = \tfrac12\,\mathrm{erfc}[\ln(|h|/h_m)/(\sqrt2\sigma)]$,
et le modèle bimodal de Durner (1994), $S_e = w\,[1+(\alpha_1|h|)^{n_1}]^{-m_1} + (1-w)\,[1+(\alpha_2|h|)^{n_2}]^{-m_2}$.
Comparer VG, Kosugi et Durner par le critère $\mathrm{AIC} = N\ln(\Phi/N) + 2k$ ($\Phi$ = somme des carrés des résidus,
$k$ = nombre de paramètres) et par sa version corrigée pour les petits échantillons, $\mathrm{AIC}_c = \mathrm{AIC} + 2k(k+1)/(N-k-1)$.
Le modèle bimodal est-il justifié avec 9 points ?

In [ ]:
def kosugi(h, tr, ts, hm, sig):
    # À COMPLÉTER
    pass

def durner(h, tr, ts, w, a1, n1, a2, n2):
    # À COMPLÉTER
    pass

g = mes[mes["sol"] == "argile"]
h, th = g["h_cm"].to_numpy(), g["theta"].to_numpy()
# ajustements et AIC

## Pour aller plus loin

* Exercice 1 : refaire l'ajustement en fixant $\theta_s$ à la valeur mesurée et $\theta_r = 0$ ; comparer les incertitudes sur $\alpha$ et $n$.
* Exercice 3 : ajouter la FPT HYPRES (Wösten et al., 1999) et comparer.
* Exercice 4 : implémenter la correction de Lenhard et al. (1991) contre le « pompage » sur plusieurs cycles successifs.